In [5]:
from scipy.io import loadmat
from pathlib import Path
import numpy as np
import pandas as pd

# ==============================
# 1. กำหนดค่าเริ่มต้น
# ==============================

fs = 48000
folder = Path.cwd()

files = sorted(folder.rglob("*.mat"))

# กำหนดตัวแปรที่ถูกต้องของแต่ละไฟล์
variable_map = {
    "Normal_0.mat": "X097_DE_time",
    "Normal_1.mat": "X098_DE_time",
    "Normal_2.mat": "X099_DE_time",
    "Normal_3.mat": "X100_DE_time",

    "IR007_0.mat": "X109_DE_time",
    "IR007_1.mat": "X110_DE_time",
    "IR007_2.mat": "X111_DE_time",
    "IR007_3.mat": "X112_DE_time",

    "B007_0.mat": "X122_DE_time",
    "B007_1.mat": "X123_DE_time",
    "B007_2.mat": "X124_DE_time",
    "B007_3.mat": "X125_DE_time",

    "OR007@6_0.mat": "X135_DE_time",
    "OR007@6_1.mat": "X136_DE_time",
    "OR007@6_2.mat": "X137_DE_time",
    "OR007@6_3.mat": "X138_DE_time"
}

results = []

# ==============================
# 2. อ่านข้อมูลทุกไฟล์
# ==============================

for file in files:

    data = loadmat(file)

    # เลือกตัวแปรตามชื่อไฟล์
    key = variable_map.get(file.name)

    if key is None or key not in data:
        print(f"Check variable: {file.name}")
        continue

    signal = data[key].flatten()
    valid = signal[np.isfinite(signal)]

    results.append({
        "File": file.name,
        "Variable": key,
        "Samples": len(signal),
        "Duration (s)": len(signal) / fs,
        "Invalid Values": np.sum(~np.isfinite(signal)),
        "Min": np.min(valid) if len(valid) else np.nan,
        "Max": np.max(valid) if len(valid) else np.nan,
        "Windows 2048": len(signal) // 2048,
        "Windows 4096": len(signal) // 4096
    })

# ==============================
# 3. แสดงผล
# ==============================

df = pd.DataFrame(results)

print("Total Files:", len(files))
print("Valid DE Files:", len(df))

display(df.round(4))

Total Files: 16
Valid DE Files: 16


,File,Variable,Samples,Duration (s),Invalid Values,Min,Max,Windows 2048,Windows 4096
0,B007_0.mat,X122_DE_time,244739,5.0987,0,-0.6246,0.7364,119,59
1,B007_1.mat,X123_DE_time,487384,10.1538,0,-0.6160,0.6755,237,118
2,B007_2.mat,X124_DE_time,486804,10.1418,0,-0.7402,0.6916,237,118
3,B007_3.mat,X125_DE_time,488545,10.1780,0,-0.6578,0.6646,238,119
4,IR007_0.mat,X109_DE_time,243938,5.0820,0,-2.8483,3.5695,119,59
5,IR007_1.mat,X110_DE_time,486224,10.1297,0,-1.5742,1.6516,237,118
6,IR007_2.mat,X111_DE_time,485643,10.1176,0,-1.6954,1.6965,237,118
7,IR007_3.mat,X112_DE_time,485643,10.1176,0,-1.6393,1.6606,237,118
8,Normal_0.mat,X097_DE_time,243938,5.0820,0,-0.2866,0.3113,119,59
9,Normal_1.mat,X098_DE_time,483903,10.0813,0,-0.3459,0.3175,236,118


In [13]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ====================================
# 1. กำหนดเงื่อนไขการทดลอง
# ====================================

fs = 48000              # Sampling Rate (Hz)
window_size = 2048      # Samples per Window
window_index = 0        # Window ที่ต้องการแสดง
motor_load = 1          # Motor Load (HP)
fault_diameter = 0.007  # Fault Diameter (inch)
same_scale = True       # ใช้สเกลแกน Y เท่ากัน

motor_speeds = {
    0: 1797,
    1: 1772,
    2: 1750,
    3: 1730
}

rpm = motor_speeds[motor_load]

# ระยะเวลาของ Window และ Sampling Interval
duration = (window_size / fs) * 1000
sampling_interval = (1 / fs) * 1e6

# ====================================
# 2. เลือกข้อมูลทั้ง 4 Class
# ====================================

selected_signals = {
    "Normal": signals[f"Normal_{motor_load}.mat"],
    "Inner Race Fault": signals[f"IR007_{motor_load}.mat"],
    "Ball Fault": signals[f"B007_{motor_load}.mat"],
    "Outer Race Fault": signals[f"OR007@6_{motor_load}.mat"]
}

# ====================================
# 3. แบ่งข้อมูลเป็น Window
# ====================================

start = window_index * window_size
end = start + window_size

windows = {
    name: signal[start:end]
    for name, signal in selected_signals.items()
}

assert all(len(w) == window_size for w in windows.values())

# แกนเวลา หน่วย milliseconds (ms)
time = (np.arange(window_size) / fs) * 1000

# Sample Index จากสัญญาณต้นฉบับ
sample_index = np.arange(start, end)

# กำหนดขอบเขตแกน Y
y_max = max(np.max(np.abs(w)) for w in windows.values())
y_max *= 1.05

# ====================================
# 4. สร้างกราฟทั้ง 4 Class
# ====================================

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=list(windows.keys())
)

colors = ["blue", "orange", "green", "red"]

for i, (name, window) in enumerate(windows.items()):

    fig.add_trace(
        go.Scatter(
            x=time,
            y=window,
            mode="lines",
            name=name,
            line=dict(color=colors[i], width=1),
            customdata=sample_index,

            # รายละเอียดเมื่อเลื่อนเมาส์ไปบนกราฟ
            hovertemplate=(
                "Sample: %{customdata}<br>"
                "Time: %{x:.4f} ms<br>"
                "Acceleration: %{y:.4f} g"
                "<extra></extra>"
            )
        ),
        row=i+1,
        col=1
    )

    # ระบุชื่อและหน่วยแกน Y
    fig.update_yaxes(
        title_text="Acceleration Amplitude (g)",
        showgrid=True,
        row=i+1,
        col=1
    )

    # ระบุชื่อและหน่วยแกน X
    fig.update_xaxes(
        title_text="Time (ms)",
        showgrid=True,
        showticklabels=True,
        row=i+1,
        col=1
    )

    # ใช้สเกลแกน Y เท่ากัน
    if same_scale:
        fig.update_yaxes(
            range=[-y_max, y_max],
            row=i+1,
            col=1
        )

# ====================================
# 5. กำหนดรายละเอียดของกราฟ
# ====================================

fig.update_layout(
    template="plotly_white",
    width=1200,
    height=1200,
    margin=dict(t=160),
    showlegend=False
)

fig.show()

In [23]:
import numpy as np
import pandas as pd
from scipy.stats import kurtosis

# ====================================
# 1. กำหนดเงื่อนไข
# ====================================

fs = 48000
window_size = 2048
motor_loads = [0, 1]

results = []

# ====================================
# 2. เลือกข้อมูลแต่ละ Class
# ====================================

for motor_load in motor_loads:

    selected_signals = {
        "Normal": signals[f"Normal_{motor_load}.mat"],
        "Inner Race Fault": signals[f"IR007_{motor_load}.mat"],
        "Ball Fault": signals[f"B007_{motor_load}.mat"],
        "Outer Race Fault": signals[f"OR007@6_{motor_load}.mat"]
    }

    # ====================================
    # 3. แบ่งข้อมูลเป็น Window
    # ====================================

    for class_name, signal in selected_signals.items():

        num_windows = len(signal) // window_size

        for i in range(num_windows):

            start = i * window_size
            end = start + window_size

            window = signal[start:end]

            # ====================================
            # 4. คำนวณค่าทางสถิติ
            # ====================================

            mean = np.mean(window)

            std = np.std(window)

            rms = np.sqrt(np.mean(window**2))

            peak = np.max(np.abs(window))

            peak_to_peak = np.max(window) - np.min(window)

            kurt = kurtosis(window, fisher=False)

            crest_factor = peak / rms if rms > 0 else np.nan

            # ====================================
            # 5. บันทึกผล
            # ====================================

            results.append({
                "Class": class_name,
                "Motor Load (HP)": motor_load,
                "Window": i + 1,
                "Mean (g)": mean,
                "SD (g)": std,
                "RMS (g)": rms,
                "Peak (g)": peak,
                "Peak-to-Peak (g)": peak_to_peak,
                "Kurtosis": kurt,
                "Crest Factor": crest_factor
            })

# ====================================
# 6. แสดงตารางผลลัพธ์
# ====================================

df_stats = pd.DataFrame(results)

print("Total Windows:", len(df_stats))

display(df_stats.round(5))

Total Windows: 1422


,Class,Motor Load (HP),Window,Mean (g),SD (g),RMS (g),Peak (g),Peak-to-Peak (g),Kurtosis,Crest Factor
0,Normal,0,1,0.01200,0.07227,0.07326,0.27287,0.49066,2.95418,3.72485
1,Normal,0,2,0.01279,0.07799,0.07904,0.22676,0.44977,2.81542,2.86917
2,Normal,0,3,0.01196,0.07337,0.07434,0.21925,0.42140,2.77824,2.94944
3,Normal,0,4,0.01028,0.07297,0.07369,0.24930,0.49671,2.93097,3.38310
4,Normal,0,5,0.01102,0.07127,0.07211,0.22468,0.42829,2.79440,3.11570
...,...,...,...,...,...,...,...,...,...,...
1417,Outer Race Fault,1,233,0.00973,1.09750,1.09754,4.93163,9.74974,7.10194,4.49333
1418,Outer Race Fault,1,234,0.00961,1.06654,1.06659,4.93831,9.85741,6.89878,4.63000
1419,Outer Race Fault,1,235,0.01266,1.08822,1.08829,4.67539,9.27941,6.59677,4.29607
1420,Outer Race Fault,1,236,0.00688,1.10100,1.10102,5.36357,10.65828,6.65305,4.87143


In [18]:
features = [
    "Mean (g)",
    "SD (g)",
    "RMS (g)",
    "Peak (g)",
    "Peak-to-Peak (g)",
    "Kurtosis",
    "Crest Factor"
]

summary = df_stats.groupby("Class")[features].mean()

display(summary.round(5))

,Mean (g),SD (g),RMS (g),Peak (g),Peak-to-Peak (g),Kurtosis,Crest Factor
Class,,,,,,,
Ball Fault,0.02351,0.14162,0.14370,0.48787,0.91470,3.02590,3.39099
Inner Race Fault,0.03944,0.37962,0.38183,1.91777,3.60612,6.89835,5.06292
Normal,0.01256,0.06760,0.06880,0.22276,0.42291,2.85791,3.23879
Outer Race Fault,0.01232,1.12330,1.12337,5.14259,10.15379,6.83217,4.60071


In [36]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# เลือกชุดข้อมูลและ Motor Load ที่ต้องการดู
selected_split = "train"
selected_load = 0

# ลำดับ Class ที่ต้องการแสดง
class_names = [
    "Normal",
    "Inner Race Fault",
    "Ball Fault",
    "Outer Race Fault"
]

# สร้างกราฟ 4 แถว: แถวละ 1 Class
fig = make_subplots(
    rows=4,
    cols=1,
    subplot_titles=class_names
)

for row, class_name in enumerate(class_names, start=1):

    # ค้นหาไฟล์ที่ตรงกับ Class และ Motor Load
    matches = [
        record
        for record in data_splits[selected_split]
        if record["class_name"] == class_name
        and record["motor_load"] == selected_load
    ]

    if len(matches) != 1:
        raise ValueError(
            f"ต้องพบข้อมูล {class_name} ที่โหลด "
            f"{selected_load} HP จำนวน 1 ไฟล์ "
            f"แต่พบ {len(matches)} ไฟล์"
        )

    record = matches[0]
    signal = record["signal"]

    # แกน X คือหมายเลขจุดข้อมูลในไฟล์ต้นฉบับ
    sample_indices = np.arange(len(signal))

    fig.add_trace(
        go.Scattergl(
            x=sample_indices,
            y=signal,
            mode="lines",
            line=dict(width=1),
            name=class_name,
            showlegend=False
        ),
        row=row,
        col=1
    )

    fig.update_xaxes(
        title_text="Sample Index",
        row=row,
        col=1
    )

    fig.update_yaxes(
        title_text="Acceleration (g)",
        row=row,
        col=1
    )

    print(
        f"Class: {class_name} | "
        f"File: {record['filename']} | "
        f"Label: {record['label']} | "
        f"จำนวนจุด: {len(signal):,}"
    )

fig.update_layout(
    title=f"Full signal of each class before dividing into windows — {selected_load} HP",
    template="plotly_white",
    height=1000,
    width=1100
)

fig.show()

Class: Normal | File: Normal_0.mat | Label: 0 | จำนวนจุด: 243,938
Class: Inner Race Fault | File: IR007_0.mat | Label: 1 | จำนวนจุด: 243,938
Class: Ball Fault | File: B007_0.mat | Label: 2 | จำนวนจุด: 244,739
Class: Outer Race Fault | File: OR007@6_0.mat | Label: 3 | จำนวนจุด: 243,538


In [34]:
import numpy as np
import pandas as pd

# 1. กำหนดหมายเลข Class ให้เหมือนกันทุกชุด
class_mapping = {
    "Normal":           {"prefix": "Normal",  "label": 0},
    "Inner Race Fault": {"prefix": "IR007",   "label": 1},
    "Ball Fault":       {"prefix": "B007",    "label": 2},
    "Outer Race Fault": {"prefix": "OR007@6", "label": 3}
}

# 2. แบ่งชุดข้อมูลตาม Motor Load
split_loads = {
    "train": [0, 1],
    "val":   [2],
    "test":  [3]
}

# ตรวจสอบว่ามีไฟล์ครบ ก่อนเริ่มจัดข้อมูล
required_files = [
    f"{info['prefix']}_{load}.mat"
    for loads in split_loads.values()
    for load in loads
    for info in class_mapping.values()
]

missing_files = [
    filename for filename in required_files
    if filename not in signals
]

if missing_files:
    raise ValueError(f"ไม่พบข้อมูลใน signals: {missing_files}")

# 3. จัดเก็บสัญญาณของแต่ละชุด
data_splits = {"train": [], "val": [], "test": []}
summary_rows = []

for split_name, loads in split_loads.items():
    for load in loads:
        for class_name, info in class_mapping.items():

            filename = f"{info['prefix']}_{load}.mat"

            # รองรับสัญญาณหนึ่งช่องที่เป็นเวกเตอร์แถวหรือคอลัมน์
            signal = np.asarray(
                signals[filename], dtype=np.float64
            ).squeeze()

            if signal.ndim != 1 or signal.size == 0:
                raise ValueError(
                    f"{filename}: ต้องเป็นสัญญาณ 1 มิติที่ไม่ว่าง"
                )

            if not np.isfinite(signal).all():
                raise ValueError(
                    f"{filename}: พบ NaN หรือ Infinity "
                    "กรุณาตรวจขั้นตอน Data Cleaning"
                )

            data_splits[split_name].append({
                "filename": filename,
                "class_name": class_name,
                "label": info["label"],
                "motor_load": load,
                "signal": signal
            })

            summary_rows.append({
                "Split": split_name,
                "File": filename,
                "Class": class_name,
                "Label": info["label"],
                "Motor Load (HP)": load,
                "Samples": signal.size
            })

# 4. ตรวจสอบว่าไม่มีไฟล์เดียวกันอยู่ข้ามชุด
file_sets = {
    split_name: {
        item["filename"] for item in records
    }
    for split_name, records in data_splits.items()
}

assert file_sets["train"].isdisjoint(file_sets["val"])
assert file_sets["train"].isdisjoint(file_sets["test"])
assert file_sets["val"].isdisjoint(file_sets["test"])

# 5. แสดงรายละเอียดและสรุปจำนวนข้อมูล
df_split = pd.DataFrame(summary_rows)
display(df_split)

split_summary = (
    df_split.groupby("Split", sort=False)
    .agg(
        Files=("File", "count"),
        Classes=("Class", "nunique"),
        Total_Samples=("Samples", "sum")
    )
    .reindex(["train", "val", "test"])
)

display(split_summary)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# เลือกโหลดเดียวกัน เพื่อเปรียบเทียบแต่ละ Class
selected_load = 0

for window_size in window_sizes:

    data = windowed_data[window_size]["train"]
    metadata = data["metadata"]

    # สร้างกราฟ 4 แถว: แถวละ 1 Class
    fig = make_subplots(
        rows=4,
        cols=1,
        subplot_titles=list(class_mapping.keys())
    )

    for row, class_name in enumerate(class_mapping, start=1):

        # ค้นหา Window ของ Class และโหลดที่เลือก
        matching_indices = metadata.index[
            (metadata["Class"] == class_name) &
            (metadata["Motor Load (HP)"] == selected_load)
        ]

        if len(matching_indices) == 0:
            raise ValueError(
                f"ไม่พบ {class_name} ที่โหลด {selected_load} HP"
            )

        # เลือก Window แรกของ Class นั้น
        idx = matching_indices[0]
        example_window = data["X"][idx]
        filename = metadata.loc[idx, "File"]
        label = data["y"][idx]

        fig.add_trace(
            go.Scatter(
                x=np.arange(window_size),
                y=example_window,
                mode="lines",
                line=dict(width=1),
                name=class_name,
                showlegend=False
            ),
            row=row,
            col=1
        )

        fig.update_yaxes(
            title_text="Acceleration (g)",
            row=row,
            col=1
        )

        fig.update_xaxes(
            title_text="ตำแหน่งข้อมูลภายใน Window",
            row=row,
            col=1
        )

        print(
            f"Window Size: {window_size} | "
            f"Class: {class_name} | "
            f"Label: {label} | "
            f"File: {filename}"
        )

    fig.update_layout(
        title=(
            f"ตัวอย่าง Window ของแต่ละ Class "
            f"ที่โหลด {selected_load} HP "
            f"— ขนาด {window_size} จุด"
        ),
        template="plotly_white",
        height=1000,
        width=1100
    )

    fig.show()

,Split,File,Class,Label,Motor Load (HP),Samples
0,train,Normal_0.mat,Normal,0,0,243938
1,train,IR007_0.mat,Inner Race Fault,1,0,243938
2,train,B007_0.mat,Ball Fault,2,0,244739
3,train,OR007@6_0.mat,Outer Race Fault,3,0,243538
4,train,Normal_1.mat,Normal,0,1,483903
5,train,IR007_1.mat,Inner Race Fault,1,1,486224
6,train,B007_1.mat,Ball Fault,2,1,487384
7,train,OR007@6_1.mat,Outer Race Fault,3,1,486804
8,val,Normal_2.mat,Normal,0,2,485063
9,val,IR007_2.mat,Inner Race Fault,1,2,485643


,Files,Classes,Total_Samples
Split,,,
train,8,4,2920468
val,4,4,1944314
test,4,4,1947795


Window Size: 2048 | Class: Normal | Label: 0 | File: Normal_0.mat
Window Size: 2048 | Class: Inner Race Fault | Label: 1 | File: IR007_0.mat
Window Size: 2048 | Class: Ball Fault | Label: 2 | File: B007_0.mat
Window Size: 2048 | Class: Outer Race Fault | Label: 3 | File: OR007@6_0.mat


Window Size: 4096 | Class: Normal | Label: 0 | File: Normal_0.mat
Window Size: 4096 | Class: Inner Race Fault | Label: 1 | File: IR007_0.mat
Window Size: 4096 | Class: Ball Fault | Label: 2 | File: B007_0.mat
Window Size: 4096 | Class: Outer Race Fault | Label: 3 | File: OR007@6_0.mat


In [ ]:

# 1. ขนาด Window ที่ต้องการทดลอง
window_sizes = [2048, 4096]

# เก็บข้อมูลแยกตาม Window Size และ Split
windowed_data = {}
summary_rows = []

for window_size in window_sizes:

    windowed_data[window_size] = {}

    for split_name, records in data_splits.items():

        X_parts = []
        y_parts = []
        metadata = []
        dropped_samples = 0

        for record in records:

            signal = record["signal"]

            # จำนวน Window ที่แบ่งได้เต็ม
            num_windows = len(signal) // window_size

            # จำนวนจุดที่นำมาใช้ และจุดที่เหลือ
            used_samples = num_windows * window_size
            dropped_samples += len(signal) - used_samples

            if num_windows == 0:
                raise ValueError(
                    f"{record['filename']}: ข้อมูลสั้นกว่า "
                    f"Window Size {window_size}"
                )

            # แบ่งสัญญาณเป็นแถว:
            # 1 แถว = 1 Window
            windows = signal[:used_samples].reshape(
                num_windows, window_size
            )

            X_parts.append(windows)

            # ทุก Window ใช้ Label ของไฟล์ต้นทาง
            labels = np.full(
                num_windows,
                record["label"],
                dtype=np.int64
            )
            y_parts.append(labels)

            # เก็บที่มาของแต่ละ Window ไว้ตรวจสอบ
            for i in range(num_windows):
                metadata.append({
                    "File": record["filename"],
                    "Class": record["class_name"],
                    "Motor Load (HP)": record["motor_load"],
                    "Window": i + 1,
                    "Start Index": i * window_size,
                    "End Index (exclusive)": (i + 1) * window_size
                })

        # รวม Window จากทุกไฟล์ภายใน Split เดียวกัน
        X = np.concatenate(X_parts, axis=0)
        y = np.concatenate(y_parts, axis=0)

        windowed_data[window_size][split_name] = {
            "X": X,
            "y": y,
            "metadata": pd.DataFrame(metadata)
        }

        summary_rows.append({
            "Window Size": window_size,
            "Split": split_name,
            "Windows": len(X),
            "X Shape": str(X.shape),
            "y Shape": str(y.shape),
            "Dropped Samples": dropped_samples
        })

# 2. แสดงจำนวน Window และรูปร่างข้อมูล
df_window_summary = pd.DataFrame(summary_rows)
display(df_window_summary)

# 3. ตรวจจำนวน Window ของแต่ละ Class หลังแบ่ง
class_count_rows = []

for window_size in window_sizes:
    for split_name in data_splits:

        metadata = windowed_data[window_size][split_name]["metadata"]
        counts = metadata["Class"].value_counts()

        for class_name in class_mapping:
            class_count_rows.append({
                "Window Size": window_size,
                "Split": split_name,
                "Class": class_name,
                "Windows": int(counts.get(class_name, 0))
            })

df_window_counts = pd.DataFrame(class_count_rows)

display(
    df_window_counts.pivot(
        index=["Window Size", "Class"],
        columns="Split",
        values="Windows"
    )[["train", "val", "test"]]
)
# แสดงตัวอย่าง Window แรกของ Training ทั้งสองขนาด
for window_size in window_sizes:

    data = windowed_data[window_size]["train"]

    example_signal = data["X"][0]
    example_label = data["y"][0]
    example_info = data["metadata"].iloc[0]

    print("=" * 60)
    print(f"Window Size: {window_size}")
    print(f"ไฟล์ต้นทาง: {example_info['File']}")
    print(f"Class: {example_info['Class']}")
    print(f"Label: {example_label}")
    print(f"Motor Load: {example_info['Motor Load (HP)']} HP")
    print(f"จำนวนจุดใน Window: {len(example_signal)}")

    print("\nค่าการสั่นสะเทือน 10 จุดแรก:")
    print(np.round(example_signal[:10], 6))
    print()

# Global Normalization

In [37]:


# เก็บข้อมูลหลัง Normalization แยกตาม Window Size
normalized_data = {}

# เก็บค่า Mean และ Standard Deviation ของแต่ละขนาด
normalization_params = {}

summary_rows = []

for window_size in window_sizes:

    # ==========================================
    # 1. ดึงข้อมูลก่อน Normalization
    # ==========================================

    X_train = windowed_data[window_size]["train"]["X"]
    X_val = windowed_data[window_size]["val"]["X"]
    X_test = windowed_data[window_size]["test"]["X"]

    # ==========================================
    # 2. คำนวณจาก Training Set เท่านั้น
    # ==========================================

    train_mean = np.mean(X_train)
    train_std = np.std(X_train)

    if train_std == 0:
        raise ValueError(
            f"Window Size {window_size}: "
            "ค่า Standard Deviation ของ Training เท่ากับ 0"
        )

    normalization_params[window_size] = {
        "mean": train_mean,
        "std": train_std
    }

    # ==========================================
    # 3. ใช้ค่าเดียวกันกับ Train, Val และ Test
    # ==========================================

    X_train_norm = (X_train - train_mean) / train_std
    X_val_norm = (X_val - train_mean) / train_std
    X_test_norm = (X_test - train_mean) / train_std

    # ==========================================
    # 4. เก็บข้อมูลและ Label
    # ==========================================

    normalized_data[window_size] = {
        "train": {
            "X": X_train_norm,
            "y": windowed_data[window_size]["train"]["y"]
        },
        "val": {
            "X": X_val_norm,
            "y": windowed_data[window_size]["val"]["y"]
        },
        "test": {
            "X": X_test_norm,
            "y": windowed_data[window_size]["test"]["y"]
        }
    }

    # ==========================================
    # 5. บันทึกผลสำหรับตรวจสอบ
    # ==========================================

    for split_name in ["train", "val", "test"]:

        X_norm = normalized_data[window_size][split_name]["X"]

        summary_rows.append({
            "Window Size": window_size,
            "Split": split_name,
            "X Shape": str(X_norm.shape),
            "Mean Before (train)": train_mean,
            "Std Before (train)": train_std,
            "Mean After": np.mean(X_norm),
            "Std After": np.std(X_norm)
        })

# แสดงตารางสรุป
df_normalization = pd.DataFrame(summary_rows)
display(df_normalization.round(5))

,Window Size,Split,X Shape,Mean Before (train),Std Before (train),Mean After,Std After
0,2048,train,"(1422, 2048)",0.02197,0.60554,-0.00000,1.00000
1,2048,val,"(947, 2048)",0.02197,0.60554,-0.01244,0.88378
2,2048,test,"(950, 2048)",0.02197,0.60554,-0.01410,0.87228
3,4096,train,"(708, 4096)",0.02195,0.60577,-0.00000,1.00000
4,4096,val,"(472, 4096)",0.02195,0.60577,-0.01240,0.88311
5,4096,test,"(474, 4096)",0.02195,0.60577,-0.01407,0.87271
